# Mô hình 5: PhoBERT Fine-tuning
## Phân loại Phản hồi Sinh viên Tiếng Việt

**Bài toán**: Phân loại cảm xúc phản hồi sinh viên (3 lớp: Negative / Neutral / Positive)

**Kiến trúc PhoBERT + Attentive Pooling**:
- PhoBERT Base (`vinai/phobert-base`) — được pre-train trên 20GB văn bản tiếng Việt
- **Attentive Pooling Head** (thay thế [CLS] token thông thường)
  - Weighted sum của tất cả hidden states
  - Giúp tập trung vào các token quan trọng hơn
- Dropout + Fully Connected (768 → 3)

> Nguyen & Nguyen (2020). *PhoBERT: Pre-trained language models for Vietnamese*. EMNLP Findings.

In [1]:
# ===================== CÀI ĐẶT THƯ VIỆN =====================
# !pip install torch transformers pandas numpy scikit-learn matplotlib seaborn

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Sử dụng thiết bị: {device}')

C:\Users\Lenovo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sử dụng thiết bị: cpu


## 1. Chuẩn bị Dữ liệu

In [2]:
# ===================== TẢI DỮ LIỆU =====================
train_df = pd.read_csv('data/train.csv')
val_df   = pd.read_csv('data/val.csv')
test_df  = pd.read_csv('data/test.csv')

CLASSES = ['Negative', 'Neutral', 'Positive']
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\nPhân phối nhãn (Train):')
print(train_df['label'].value_counts().sort_index())

Train: 1470 | Val: 209 | Test: 421

Phân phối nhãn (Train):
label
0    490
1    490
2    490
Name: count, dtype: int64


In [3]:
# ===================== TOKENIZER PHOBERT =====================
print('Đang tải PhoBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
print('Tokenizer đã sẵn sàng!')

MAX_LEN    = 64
BATCH_SIZE = 16  # Nhỏ hơn vì PhoBERT nặng hơn

class BERTFeedbackDataset(Dataset):
    """
    Dataset dùng PhoBERT tokenizer.
    Trả về: input_ids, attention_mask, labels
    """
    def __init__(self, df, tokenizer, max_len=64):
        self.texts     = df['text'].values
        self.labels    = df['label'].values
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(BERTFeedbackDataset(train_df, tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(BERTFeedbackDataset(val_df,   tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE)
test_loader  = DataLoader(BERTFeedbackDataset(test_df,  tokenizer, MAX_LEN),
                          batch_size=BATCH_SIZE)
print('DataLoaders sẵn sàng!')

Đang tải PhoBERT tokenizer...


Tokenizer đã sẵn sàng!
DataLoaders sẵn sàng!


## 2. Kiến trúc Mô hình PhoBERT + Attentive Pooling

In [4]:
# ===================== MÔ HÌNH PHOBERT + ATTENTIVE POOLING =====================
class AttentivePooling(nn.Module):
    """
    Attentive Pooling thay thế [CLS] token.
    Tính trọng số attention cho từng token, sau đó weighted sum.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.v = nn.Parameter(torch.rand(hidden_dim))
        self.w = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x, mask=None):
        # x: [batch, seq_len, hidden_dim]
        u      = torch.tanh(self.w(x))             # [batch, seq_len, hidden_dim]
        score  = torch.matmul(u, self.v)            # [batch, seq_len]
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(score, dim=1)       # [batch, seq_len]
        return torch.sum(x * weights.unsqueeze(-1), dim=1)  # [batch, hidden_dim]


class PhoBERTClassifier(nn.Module):
    """
    PhoBERT fine-tuning với custom Attentive Pooling head.
    Thay vì chỉ dùng [CLS] token, sử dụng weighted sum của tất cả hidden states.
    """
    def __init__(self, model_name='vinai/phobert-base', output_dim=3, dropout=0.1):
        super().__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        hidden_dim   = self.phobert.config.hidden_size  # 768
        self.attention_pool = AttentivePooling(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_ids, attention_mask):
        outputs         = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden     = outputs.last_hidden_state    # [batch, seq_len, 768]
        pooled          = self.attention_pool(last_hidden, mask=attention_mask)
        return self.fc(self.dropout(pooled))

print('Đang tải PhoBERT model...')
model = PhoBERTClassifier(model_name='vinai/phobert-base', output_dim=3, dropout=0.1).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Tổng tham số: {total_params:,}')
print(f'Kích thước mô hình: {total_params * 4 / 1024**2:.2f} MB')

Đang tải PhoBERT model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1318.97it/s, Materializing param=pooler.dense.weight]                               
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tổng tham số: 135,591,939
Kích thước mô hình: 517.24 MB


## 3. Huấn luyện (Fine-tuning)

In [5]:
# ===================== TRAINING LOOP (FINE-TUNING) =====================
# PhoBERT dùng AdamW, lr nhỏ hơn (2e-5), và chỉ cần 3-5 epochs
EPOCHS   = 15
LR       = 2e-5
PATIENCE = 3

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print('='*60)
print('Bắt đầu Fine-tuning PhoBERT...')
print('='*60)
start_time = time.time()

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0
    train_correct, train_total = 0, 0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward(); optimizer.step()
        train_loss += loss.item()
        train_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
        train_total += labels.size(0)
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total

    # --- Validate ---
    model.eval()
    val_loss = 0
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            logits = model(input_ids, attention_mask)
            val_loss += criterion(logits, labels).item()
            val_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
            val_total += labels.size(0)
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    print(f'Epoch {epoch+1:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'PhoBERT.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            pass
        # print(f'Early stopping tại epoch {epoch+1}')
        # # break # Disabled as requested

total_time = time.time() - start_time
print(f'\nThời gian huấn luyện: {total_time:.1f}s ({total_time/60:.1f} phút)')
model.load_state_dict(torch.load('PhoBERT.pt', map_location=device))
print('Đã load mô hình tốt nhất!')

Bắt đầu Fine-tuning PhoBERT...
Epoch  1/15 | Train Loss: 0.6446 | Val Loss: 0.0874
Epoch  2/15 | Train Loss: 0.0585 | Val Loss: 0.0130
Epoch  3/15 | Train Loss: 0.0457 | Val Loss: 0.0146
Epoch  4/15 | Train Loss: 0.0203 | Val Loss: 0.0062
Epoch  5/15 | Train Loss: 0.0078 | Val Loss: 0.0039
Epoch  6/15 | Train Loss: 0.0048 | Val Loss: 0.0028
Epoch  7/15 | Train Loss: 0.0039 | Val Loss: 0.0023
Epoch  8/15 | Train Loss: 0.0069 | Val Loss: 0.0021


KeyboardInterrupt: 

## 4. Đánh giá trên Tập Test

In [ ]:
from IPython.display import display
# ===================== ĐÁNH GIÁ =====================
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        preds = torch.argmax(model(input_ids, attention_mask), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print('\n' + '='*60)
print('KẾT QUẢ MÔ HÌNH: PHOBERT + ATTENTIVE POOLING')
print('='*60)
print(classification_report(all_labels, all_preds, target_names=CLASSES))

# Lưu bảng metrics vào CSV và in ra DataFrame
report_dict = classification_report(all_labels, all_preds, target_names=CLASSES, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df)
report_df.to_csv('PhoBERT_metrics.csv', encoding='utf-8-sig')

## 5. Trực quan hóa Kết quả

In [ ]:
# ===================== VISUALIZATION =====================
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_title('PhoBERT - Loss Curve (Fine-tuning)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True)

# Accuracy curve
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'],   label='Val Acc',   marker='s')
axes[1].set_title('PhoBERT - Accuracy Curve')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True)


cm = confusion_matrix(all_labels, all_preds, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[2])
axes[2].set_title('PhoBERT - Confusion Matrix')
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('PhoBERT_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Đã lưu kết quả vào PhoBERT_results.png')